Author: Amanda Baright
Purpose: ST 554 Project 2
Date: 03.25.2026

# ST 554 Project 2

## Part I: Using my_class.py

This section of the proect will focus on using an air quality dataset to test my created python class called `SparkDataCheck` in a python file called`my_class.py`. This python class will work on a Spark SQL style data frame and has a few methods. There are two `@classmethods` that create instances from reading in csv files and reading in pandas dataframes. There are also a few validation methods that append Boolean columns to the dataframe and serve to check if the numeric columns fall under a user specified range (`check_numeric_range`), if string columns fall within a user specified set of levels (`check_string_levels`), and if there are missing values in a column (`check_missing`). The class then wraps up with two summarization methods that report the min and max of numeric columns (`report_min_max`) and reports the counts associated with one or two string columns (`report_counts`). Both summarization methods account for a user provided grouping variable and return a pandas dataframe of the summarization.

The following section will focus on using this air quality dataset (https://www4.stat.ncsu.edu/online/datasets/air.csv) to test the `SparkDataCheck` class and its methods. We first need to import our packages and create a Spark Session.

In [36]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql import SparkSession
import numpy as np
import pandas as pd
import pyspark.pandas as ps

spark = SparkSession.builder.appName('my_app').config("spark.sql.ansi.enabled", "false").getOrCreate()

We then need to import the `my_class` script so that it can be access in this `.ipynb` file.

In [37]:
from my_class import SparkDataCheck

Now that we imported in `SparkDataCheck`, we want to read in the air quality data as a csv file. For this, the csv file will need to be downloaded and uploaded into the JupyterHub. We will then use the class methods to create an instance of the class from the csv file.

In [38]:
air_quality = SparkDataCheck.from_csv(spark, 'air.csv')

print("Data successfully loaded. Schema: ")
air_quality.df.printSchema()

Data successfully loaded. Schema: 
root
 |-- _c0: integer (nullable = true)
 |-- Date: string (nullable = true)
 |-- Time: timestamp (nullable = true)
 |-- CO(GT): double (nullable = true)
 |-- PT08.S1(CO): integer (nullable = true)
 |-- NMHC(GT): integer (nullable = true)
 |-- C6H6(GT): double (nullable = true)
 |-- PT08.S2(NMHC): integer (nullable = true)
 |-- NOx(GT): integer (nullable = true)
 |-- PT08.S3(NOx): integer (nullable = true)
 |-- NO2(GT): integer (nullable = true)
 |-- PT08.S4(NO2): integer (nullable = true)
 |-- PT08.S5(O3): integer (nullable = true)
 |-- T: double (nullable = true)
 |-- RH: double (nullable = true)
 |-- AH: double (nullable = true)



For ease of use, we want to do some basic data cleaning before our examples to rename our columns, convert the `-200` values to `Null` and we can create a binary variable with low vs high temperature using the median as the middle point to determine the binary category.

In [39]:
# Remap the column names
rename_map = {
    "_c0": "id",
    "CO(GT)": "co_gt",
    "PT08.S1(CO)": "sensor_co",
    "NMHC(GT)": "nmhc_gt",
    "C6H6(GT)": "benzene_gt",
    "PT08.S2(NMHC)": "sensor_nmhc",
    "NOx(GT)": "nox_gt",
    "PT08.S3(NOx)": "sensor_nox",
    "NO2(GT)": "no2_gt",
    "PT08.S4(NO2)": "sensor_no2",
    "PT08.S5(O3)": "sensor_o3",
    "T": "temp",
    "RH": "rel_humidity",
    "AH": "abs_humidity"
}

# Now apply the new names to the old names, and also replace -200 with None
for old, new in rename_map.items():
    if old in air_quality.df.columns:
        air_quality.df = air_quality.df.withColumn(
            new, 
            F.when(F.col(f"`{old}`") == -200, None).otherwise(F.col(f"`{old}`"))
        )
        if old != new:
            air_quality.df = air_quality.df.drop(old)
            
# Now print and check the new schema
print("Cleaned Schema:")
air_quality.df.printSchema()

Cleaned Schema:
root
 |-- Date: string (nullable = true)
 |-- Time: timestamp (nullable = true)
 |-- id: integer (nullable = true)
 |-- co_gt: double (nullable = true)
 |-- sensor_co: integer (nullable = true)
 |-- nmhc_gt: integer (nullable = true)
 |-- benzene_gt: double (nullable = true)
 |-- sensor_nmhc: integer (nullable = true)
 |-- nox_gt: integer (nullable = true)
 |-- sensor_nox: integer (nullable = true)
 |-- no2_gt: integer (nullable = true)
 |-- sensor_no2: integer (nullable = true)
 |-- sensor_o3: integer (nullable = true)
 |-- temp: double (nullable = true)
 |-- rel_humidity: double (nullable = true)
 |-- abs_humidity: double (nullable = true)



We now want to create a binary variable we can use to test the string method. Here we will find the median temperature and create a binary low vs high category to determine if the given temperature is lower or higher than the median value.

In [40]:
median_temp = air_quality.df.approxQuantile("temp", [0.5], 0.01)[0]
print(f"The median temperature is: {median_temp}")

# Create the binary variable
air_quality.df = air_quality.df.withColumn(
    "temp_category",
    F.when(F.col("temp").isNull(), None)
     .when(F.col("temp") <= median_temp, "Low")
     .otherwise("High")
)

The median temperature is: 17.7


In case we need a second binary variable, we can look at the `benzene_category` which will be a high vs low binary variable, where the median value is used to determine if an observation is high or low.

In [41]:
median_benzene = air_quality.df.approxQuantile("benzene_gt", [0.5], 0.01)[0]
print(f"The median benzene is: {median_benzene}")

# Create the binary variable
air_quality.df = air_quality.df.withColumn(
    "benzene_category",
    F.when(F.col("benzene_gt").isNull(), None)
     .when(F.col("benzene_gt") <= median_benzene, "Low")
     .otherwise("High")
)

The median benzene is: 8.2


Now we want to provide a few examples (4-5 examples) for each method used in `my_class.py` using the instance of the class from the csv file.

### Method 1: check_numeric_range

This first section of examples will be used to test the `check_numeric_range` method. Here we should expect to see appended Boolean columns when a numeric column is provided. If a non-numeric column is provided, then the dataframe is returned without modification.

#### Example 1

This first example will check if the temperature is within a specific range of two bounds (0 to 12).

In [13]:
air_quality.check_numeric_range("temp", 0, 12).df.show(10)

+---------+-------------------+---+-----+---------+-------+----------+-----------+------+----------+------+----------+---------+----+------------+------------+-------------+----------------+--------------+
|     Date|               Time| id|co_gt|sensor_co|nmhc_gt|benzene_gt|sensor_nmhc|nox_gt|sensor_nox|no2_gt|sensor_no2|sensor_o3|temp|rel_humidity|abs_humidity|temp_category|benzene_category|temp_in_bounds|
+---------+-------------------+---+-----+---------+-------+----------+-----------+------+----------+------+----------+---------+----+------------+------------+-------------+----------------+--------------+
|3/10/2004|2026-03-26 18:00:00|  0|  2.6|     1360|    150|      11.9|       1046|   166|      1056|   113|      1692|     1268|13.6|        48.9|      0.7578|          Low|            High|         false|
|3/10/2004|2026-03-26 19:00:00|  1|  2.0|     1292|    112|       9.4|        955|   103|      1174|    92|      1559|      972|13.3|        47.7|      0.7255|          Low|   

26/03/26 21:19:21 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ambarigh@ncsu.edu/air.csv


#### Example 2

We now want an example that checks Relative Humidity with only an upper bound of (50).

In [14]:
air_quality.check_numeric_range("rel_humidity", upper = 50).df.show(10)

+---------+-------------------+---+-----+---------+-------+----------+-----------+------+----------+------+----------+---------+----+------------+------------+-------------+----------------+--------------+----------------------+
|     Date|               Time| id|co_gt|sensor_co|nmhc_gt|benzene_gt|sensor_nmhc|nox_gt|sensor_nox|no2_gt|sensor_no2|sensor_o3|temp|rel_humidity|abs_humidity|temp_category|benzene_category|temp_in_bounds|rel_humidity_in_bounds|
+---------+-------------------+---+-----+---------+-------+----------+-----------+------+----------+------+----------+---------+----+------------+------------+-------------+----------------+--------------+----------------------+
|3/10/2004|2026-03-26 18:00:00|  0|  2.6|     1360|    150|      11.9|       1046|   166|      1056|   113|      1692|     1268|13.6|        48.9|      0.7578|          Low|            High|         false|                  true|
|3/10/2004|2026-03-26 19:00:00|  1|  2.0|     1292|    112|       9.4|        955|  

26/03/26 21:21:04 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ambarigh@ncsu.edu/air.csv


#### Example 3

Now we want to show an example where we check the benzene levels to have a lower bound of 5.

In [15]:
air_quality.check_numeric_range("benzene_gt", lower = 5).df.show(10)

+---------+-------------------+---+-----+---------+-------+----------+-----------+------+----------+------+----------+---------+----+------------+------------+-------------+----------------+--------------+----------------------+--------------------+
|     Date|               Time| id|co_gt|sensor_co|nmhc_gt|benzene_gt|sensor_nmhc|nox_gt|sensor_nox|no2_gt|sensor_no2|sensor_o3|temp|rel_humidity|abs_humidity|temp_category|benzene_category|temp_in_bounds|rel_humidity_in_bounds|benzene_gt_in_bounds|
+---------+-------------------+---+-----+---------+-------+----------+-----------+------+----------+------+----------+---------+----+------------+------------+-------------+----------------+--------------+----------------------+--------------------+
|3/10/2004|2026-03-26 18:00:00|  0|  2.6|     1360|    150|      11.9|       1046|   166|      1056|   113|      1692|     1268|13.6|        48.9|      0.7578|          Low|            High|         false|                  true|                true|


26/03/26 21:22:35 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ambarigh@ncsu.edu/air.csv


#### Example 4

Now we want to try an example where we supply the `check_numeric_range` method a non-numeric column, like the `temp_category` variable.

In [16]:
air_quality.check_numeric_range("temp_category").df.show(10)

Message: Column 'temp_category' is non-numeric.
+---------+-------------------+---+-----+---------+-------+----------+-----------+------+----------+------+----------+---------+----+------------+------------+-------------+----------------+--------------+----------------------+--------------------+
|     Date|               Time| id|co_gt|sensor_co|nmhc_gt|benzene_gt|sensor_nmhc|nox_gt|sensor_nox|no2_gt|sensor_no2|sensor_o3|temp|rel_humidity|abs_humidity|temp_category|benzene_category|temp_in_bounds|rel_humidity_in_bounds|benzene_gt_in_bounds|
+---------+-------------------+---+-----+---------+-------+----------+-----------+------+----------+------+----------+---------+----+------------+------------+-------------+----------------+--------------+----------------------+--------------------+
|3/10/2004|2026-03-26 18:00:00|  0|  2.6|     1360|    150|      11.9|       1046|   166|      1056|   113|      1692|     1268|13.6|        48.9|      0.7578|          Low|            High|         fal

26/03/26 21:27:19 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ambarigh@ncsu.edu/air.csv


### Method 2: check_string_levels method

Next we want to test the `check_string_levels` method.

#### Example 1

The first example can be used to check the string levels of the `temp_category` variable. Here we supply both levels to the method.

In [17]:
air_quality.check_string_levels("temp_category", ["High", "Low"]).df.show(10)

+---------+-------------------+---+-----+---------+-------+----------+-----------+------+----------+------+----------+---------+----+------------+------------+-------------+----------------+--------------+----------------------+--------------------+-------------------------+
|     Date|               Time| id|co_gt|sensor_co|nmhc_gt|benzene_gt|sensor_nmhc|nox_gt|sensor_nox|no2_gt|sensor_no2|sensor_o3|temp|rel_humidity|abs_humidity|temp_category|benzene_category|temp_in_bounds|rel_humidity_in_bounds|benzene_gt_in_bounds|temp_category_valid_level|
+---------+-------------------+---+-----+---------+-------+----------+-----------+------+----------+------+----------+---------+----+------------+------------+-------------+----------------+--------------+----------------------+--------------------+-------------------------+
|3/10/2004|2026-03-26 18:00:00|  0|  2.6|     1360|    150|      11.9|       1046|   166|      1056|   113|      1692|     1268|13.6|        48.9|      0.7578|          Low

26/03/26 21:35:43 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ambarigh@ncsu.edu/air.csv


#### Example 2

Now what if we want to supply just one level of `benzene_category` to the method.

In [18]:
air_quality.check_string_levels("benzene_category", ["Low"]).df.show(10)

+---------+-------------------+---+-----+---------+-------+----------+-----------+------+----------+------+----------+---------+----+------------+------------+-------------+----------------+--------------+----------------------+--------------------+-------------------------+----------------------------+
|     Date|               Time| id|co_gt|sensor_co|nmhc_gt|benzene_gt|sensor_nmhc|nox_gt|sensor_nox|no2_gt|sensor_no2|sensor_o3|temp|rel_humidity|abs_humidity|temp_category|benzene_category|temp_in_bounds|rel_humidity_in_bounds|benzene_gt_in_bounds|temp_category_valid_level|benzene_category_valid_level|
+---------+-------------------+---+-----+---------+-------+----------+-----------+------+----------+------+----------+---------+----+------------+------------+-------------+----------------+--------------+----------------------+--------------------+-------------------------+----------------------------+
|3/10/2004|2026-03-26 18:00:00|  0|  2.6|     1360|    150|      11.9|       1046|   

26/03/26 21:37:36 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ambarigh@ncsu.edu/air.csv


#### Example 3

Now what if we want to see if this method will work with one of our appended Boolean columns.

In [20]:
air_quality.check_string_levels("temp_in_bounds", ["true"]).df.show(10)

Message: Column 'temp_in_bounds' is not a string column.
+---------+-------------------+---+-----+---------+-------+----------+-----------+------+----------+------+----------+---------+----+------------+------------+-------------+----------------+--------------+----------------------+--------------------+-------------------------+----------------------------+
|     Date|               Time| id|co_gt|sensor_co|nmhc_gt|benzene_gt|sensor_nmhc|nox_gt|sensor_nox|no2_gt|sensor_no2|sensor_o3|temp|rel_humidity|abs_humidity|temp_category|benzene_category|temp_in_bounds|rel_humidity_in_bounds|benzene_gt_in_bounds|temp_category_valid_level|benzene_category_valid_level|
+---------+-------------------+---+-----+---------+-------+----------+-----------+------+----------+------+----------+---------+----+------------+------------+-------------+----------------+--------------+----------------------+--------------------+-------------------------+----------------------------+
|3/10/2004|2026-03-26 18:00:

26/03/26 21:39:06 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ambarigh@ncsu.edu/air.csv


#### Example 4

We know that `Date` is also a string, so let's check which columns are for the date "3/10/2004".

In [21]:
air_quality.check_string_levels("Date", ["3/10/2004"]).df.show(10)

+---------+-------------------+---+-----+---------+-------+----------+-----------+------+----------+------+----------+---------+----+------------+------------+-------------+----------------+--------------+----------------------+--------------------+-------------------------+----------------------------+----------------+
|     Date|               Time| id|co_gt|sensor_co|nmhc_gt|benzene_gt|sensor_nmhc|nox_gt|sensor_nox|no2_gt|sensor_no2|sensor_o3|temp|rel_humidity|abs_humidity|temp_category|benzene_category|temp_in_bounds|rel_humidity_in_bounds|benzene_gt_in_bounds|temp_category_valid_level|benzene_category_valid_level|Date_valid_level|
+---------+-------------------+---+-----+---------+-------+----------+-----------+------+----------+------+----------+---------+----+------------+------------+-------------+----------------+--------------+----------------------+--------------------+-------------------------+----------------------------+----------------+
|3/10/2004|2026-03-26 18:00:00|  0

26/03/26 21:41:05 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ambarigh@ncsu.edu/air.csv


#### Example 5

Now we can do a final example with a numeric column to make sure that the message is printed with a numeric column.

In [22]:
air_quality.check_string_levels("co_gt", ["1.2"]).df.show(10)

Message: Column 'co_gt' is not a string column.
+---------+-------------------+---+-----+---------+-------+----------+-----------+------+----------+------+----------+---------+----+------------+------------+-------------+----------------+--------------+----------------------+--------------------+-------------------------+----------------------------+----------------+
|     Date|               Time| id|co_gt|sensor_co|nmhc_gt|benzene_gt|sensor_nmhc|nox_gt|sensor_nox|no2_gt|sensor_no2|sensor_o3|temp|rel_humidity|abs_humidity|temp_category|benzene_category|temp_in_bounds|rel_humidity_in_bounds|benzene_gt_in_bounds|temp_category_valid_level|benzene_category_valid_level|Date_valid_level|
+---------+-------------------+---+-----+---------+-------+----------+-----------+------+----------+------+----------+---------+----+------------+------------+-------------+----------------+--------------+----------------------+--------------------+-------------------------+----------------------------+----

26/03/26 21:43:27 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ambarigh@ncsu.edu/air.csv


### Method 3: check_missing method

Now we can try a few examples to test the `check_missing` method. Here we only do two, per Dr. Post's Slack message.

#### Example 1

First we want to check for missing in the benzene column.

In [24]:
air_quality.check_missing("benzene_gt").df.show(5)

+---------+-------------------+---+-----+---------+-------+----------+-----------+------+----------+------+----------+---------+----+------------+------------+-------------+----------------+--------------+----------------------+--------------------+-------------------------+----------------------------+----------------+------------------+
|     Date|               Time| id|co_gt|sensor_co|nmhc_gt|benzene_gt|sensor_nmhc|nox_gt|sensor_nox|no2_gt|sensor_no2|sensor_o3|temp|rel_humidity|abs_humidity|temp_category|benzene_category|temp_in_bounds|rel_humidity_in_bounds|benzene_gt_in_bounds|temp_category_valid_level|benzene_category_valid_level|Date_valid_level|benzene_gt_is_null|
+---------+-------------------+---+-----+---------+-------+----------+-----------+------+----------+------+----------+---------+----+------------+------------+-------------+----------------+--------------+----------------------+--------------------+-------------------------+----------------------------+--------------

26/03/26 21:48:53 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ambarigh@ncsu.edu/air.csv


#### Example 2

Now we might want to check missing with a string column.

In [25]:
air_quality.check_missing("temp_category").df.show(5)

+---------+-------------------+---+-----+---------+-------+----------+-----------+------+----------+------+----------+---------+----+------------+------------+-------------+----------------+--------------+----------------------+--------------------+-------------------------+----------------------------+----------------+------------------+---------------------+
|     Date|               Time| id|co_gt|sensor_co|nmhc_gt|benzene_gt|sensor_nmhc|nox_gt|sensor_nox|no2_gt|sensor_no2|sensor_o3|temp|rel_humidity|abs_humidity|temp_category|benzene_category|temp_in_bounds|rel_humidity_in_bounds|benzene_gt_in_bounds|temp_category_valid_level|benzene_category_valid_level|Date_valid_level|benzene_gt_is_null|temp_category_is_null|
+---------+-------------------+---+-----+---------+-------+----------+-----------+------+----------+------+----------+---------+----+------------+------------+-------------+----------------+--------------+----------------------+--------------------+-------------------------

26/03/26 21:49:49 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/03/26 21:49:49 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ambarigh@ncsu.edu/air.csv


### Method 4: report_min_max method

Now we can move into our summarization methods and start with the `report_min_max` method that can report the min and max of a user specified variable and has the option to include a grouping variable. These methods will return a pandas data frame.

#### Example 1

The first example allows us to look at the min and max of benzene without any grouping.

In [27]:
air_quality.report_min_max("benzene_gt")

,benzene_gt_min,benzene_gt_max
0,0.1,63.7


#### Example 2

Now we might want to see how these min and max change with a grouping variable, like our `temp_category` variable.

In [28]:
air_quality.report_min_max("benzene_gt", "temp_category")

,temp_category,benzene_gt_min,benzene_gt_max
0,High,0.5,52.1
1,Low,0.1,63.7
2,None,NaN,NaN


#### Example 3

Now we might want to try grouping by a Boolean column, like the appended `temp_in_bounds` column.

In [29]:
air_quality.report_min_max("temp", "temp_in_bounds")

,temp_in_bounds,temp_min,temp_max
0,None,NaN,NaN
1,True,0.0,12.0
2,False,-1.9,44.6


#### Example 4

Now we might want to test what happens if no column is supplied. Here we see that the min and max of all numeric columns are supplied as expected.

In [30]:
air_quality.report_min_max()

26/03/26 21:57:49 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: 
 Schema: _c0
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ambarigh@ncsu.edu/air.csv


,id_min,id_max,co_gt_min,co_gt_max,sensor_co_min,sensor_co_max,nmhc_gt_min,nmhc_gt_max,benzene_gt_min,benzene_gt_max,...,sensor_no2_min,sensor_no2_max,sensor_o3_min,sensor_o3_max,temp_min,temp_max,rel_humidity_min,rel_humidity_max,abs_humidity_min,abs_humidity_max
0,0,9356,0.1,11.9,647,2040,7,1189,0.1,63.7,...,551,2775,221,2523,-1.9,44.6,9.2,88.7,0.1847,2.231


#### Example 5

Finally, we might want to test that a message is printed if a non-numeric column is supplied.

In [31]:
air_quality.report_min_max("temp_category")

Message: Column 'temp_category' is non-numeric.


### Method 5: report_counts method

Now we want to test the `report_counts` method.

#### Example 1

Since we have a few binary variables now, we might want to check the counts of each level for the `temp_category` column.

In [32]:
air_quality.report_counts("temp_category")

,temp_category,count
0,High,4497
1,Low,4494
2,None,366


#### Example 2

Now we want to check this method with two user supplied columns, `temp_category` and `benzene_category`.

In [33]:
air_quality.report_counts("temp_category", "benzene_category")

,temp_category,benzene_category,count
0,None,None,366
1,High,Low,1777
2,Low,Low,2725
3,Low,High,1769
4,High,High,2720


#### Example 3

The method should only allow you to supply to separate arguments, let's test this.

In [34]:
air_quality.report_counts("temp_category", "benzene_category", "Date")

TypeError: report_counts() takes from 2 to 3 positional arguments but 4 were given

#### Example 4

Now we want to see what happens if a numeric column is provided. The instructions did not specify if the summarization was to still be returned.

In [42]:
air_quality.report_counts("benzene_gt")

Message: Column 'benzene_gt' is numeric/non-string.


,benzene_gt,count
0,15.5,27
1,14.9,22
2,13.4,42
3,26.7,6
4,15.4,25
...,...,...
403,39.2,1
404,46.3,1
405,16.4,24
406,29.9,2


### Create an instance from the pandas dataframe

Now that we did a few examples using the instance created from the csv file. We want to read in the same data set using pandas and create an instance of this class from the pandas data frame.

In [13]:
air_raw = pd.read_csv("air.csv")
air_raw.head(5) # Check that it got read in

,Unnamed: 0,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
0,0,3/10/2004,18:00:00,2.6,1360,150,11.9,1046,166,1056,113,1692,1268,13.6,48.9,0.7578
1,1,3/10/2004,19:00:00,2.0,1292,112,9.4,955,103,1174,92,1559,972,13.3,47.7,0.7255
2,2,3/10/2004,20:00:00,2.2,1402,88,9.0,939,131,1140,114,1555,1074,11.9,54.0,0.7502
3,3,3/10/2004,21:00:00,2.2,1376,80,9.2,948,172,1092,122,1584,1203,11.0,60.0,0.7867
4,4,3/10/2004,22:00:00,1.6,1272,51,6.5,836,131,1205,116,1490,1110,11.2,59.6,0.7888


In [16]:
# Create an instance using our class
air_pdf = SparkDataCheck.from_pandas(spark, air_raw)
air_pdf.df.show(5) # Another quick check to see top 5 rows

+----------+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|Unnamed: 0|     Date|    Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|
+----------+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|         0|3/10/2004|18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|
|         1|3/10/2004|19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|
|         2|3/10/2004|20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|
|         3|3/10/2004|21:00:00|   2.2|       1376|      80|     9.2|        

Again, we need some data cleaning.

In [18]:
# Remap the column names
rename_map = {
    "_c0": "id",
    "CO(GT)": "co_gt",
    "PT08.S1(CO)": "sensor_co",
    "NMHC(GT)": "nmhc_gt",
    "C6H6(GT)": "benzene_gt",
    "PT08.S2(NMHC)": "sensor_nmhc",
    "NOx(GT)": "nox_gt",
    "PT08.S3(NOx)": "sensor_nox",
    "NO2(GT)": "no2_gt",
    "PT08.S4(NO2)": "sensor_no2",
    "PT08.S5(O3)": "sensor_o3",
    "T": "temp",
    "RH": "rel_humidity",
    "AH": "abs_humidity"
}

# Now apply the new names to the old names, and also replace -200 with None
for old, new in rename_map.items():
    if old in air_pdf.df.columns:
        air_pdf.df = air_pdf.df.withColumn(
            new, 
            F.when(F.col(f"`{old}`") == -200, None).otherwise(F.col(f"`{old}`"))
        )
        if old != new:
            air_pdf.df = air_pdf.df.drop(old)
            
# Create temperature binary variable
median_temp = air_quality.df.approxQuantile("temp", [0.5], 0.01)[0]
print(f"The median temperature is: {median_temp}")

# Create the binary variable
air_pdf.df = air_pdf.df.withColumn(
    "temp_category",
    F.when(F.col("temp").isNull(), None)
     .when(F.col("temp") <= median_temp, "Low")
     .otherwise("High")
)
            
# Now print and check the new schema
print("Cleaned Schema:")
air_pdf.df.printSchema()

The median temperature is: 17.7
Cleaned Schema:
root
 |-- Unnamed: 0: long (nullable = true)
 |-- Date: string (nullable = true)
 |-- Time: string (nullable = true)
 |-- co_gt: double (nullable = true)
 |-- sensor_co: long (nullable = true)
 |-- nmhc_gt: long (nullable = true)
 |-- benzene_gt: double (nullable = true)
 |-- sensor_nmhc: long (nullable = true)
 |-- nox_gt: long (nullable = true)
 |-- sensor_nox: long (nullable = true)
 |-- no2_gt: long (nullable = true)
 |-- sensor_no2: long (nullable = true)
 |-- sensor_o3: long (nullable = true)
 |-- temp: double (nullable = true)
 |-- rel_humidity: double (nullable = true)
 |-- abs_humidity: double (nullable = true)
 |-- temp_category: string (nullable = true)



### Example 5: Using report_min_max with grouping

Now that we have a pandas instance of the air quality data set, we can use the `report_min_max` method to determine the min and max of benzene across a grouping variable (`temp_category`).

In [19]:
air_pdf.report_min_max("benzene_gt", group_var = "temp_category")

,temp_category,benzene_gt_min,benzene_gt_max
0,High,0.5,52.1
1,Low,0.1,63.7
2,None,NaN,NaN


## Part II: NFL analysis using spark